# 03 — Trevisan's Taxonomy and the Consistency Principle

Synthesise the opening arc of Module 4: turn the two examples you built into a *law* every quantum optimal transport must obey, then place the competing definitions of QOT on one map.

**Prerequisites:** `04/01_diagonal_collapse`, `04/02_noncommuting_same_diagonal`.

**What you'll be able to do**
- State the **diagonal-collapse consistency principle**: a legitimate quantum OT must reduce to classical OT whenever the two states commute.
- Confirm the principle numerically on a commuting (diagonal) pair, where classical OT gives the answer every candidate must match.
- Read **Trevisan's taxonomy** — the landscape of non-equivalent QOT formulations — and see why this course follows the coupling path.
- **Open** the quantum-optimal-transport row of the running classical ↔ quantum dictionary.

You arrive with two examples in hand. In `04/01` you watched classical OT call $|+\rangle\langle+|$ and $I/2$ identical because they share a $Z$-diagonal; in `04/02` you sharpened that into a *non-commuting* pair, $|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$, and read the commutator's Frobenius norm as a dial measuring exactly how much the diagonal discards. This synthesis does two things with that work. First it extracts the *positive* lesson hiding inside the negative examples — a requirement that pins down what any honest quantum OT must do on the easy (commuting) cases. Then it steps back to the whole field and lays the competing definitions side by side, so you can see where the object this module builds sits among them.

In [ ]:
import numpy as np
import ot

from qot_course import viz
from qot_course.quantum_ot.preview import (
    commutativity_norm,
    diagonal_in_computational_basis,
)
from qot_course.transport.discrete import cityblock_cost

np.random.seed(42)
viz.use_course_style()

## 1. The consistency principle — classical OT is the commuting boundary

Both examples so far were *negative*: they showed classical OT failing on quantum states. Turn them over and a *positive* requirement appears. The trouble in `04/01`–`04/02` was always the same — coherence the $Z$-diagonal cannot see, certified by a non-zero commutator. So ask the opposite question: *when does that trouble vanish entirely?*

It vanishes exactly when the two states **commute**. If $[\rho, \sigma] = 0$ then a single basis diagonalises both at once, and in that shared eigenbasis each one genuinely *is* a probability vector — there are no off-diagonal coherences left to discard. On commuting states the diagonal reduction loses nothing, and classical OT is not blind but *correct*. The commutator norm you built in `04/02` is the gauge of this: it reads exactly zero precisely when the pair is, in the relevant sense, classical.

That observation is strong enough to become a design rule. Any honest definition of quantum optimal transport must agree with classical OT on the cases where classical OT is already right:

> **Diagonal-collapse consistency principle.** Whenever two density matrices commute (share an eigenbasis), a legitimate quantum optimal transport must reduce to *classical* optimal transport on their common eigenvalues. Classical OT is the **commuting boundary** of quantum OT.

This is not a theorem we prove here; it is the **baseline every candidate must pass** — the first thing Trevisan's survey checks of any proposed formulation. A definition that disagreed with classical OT on commuting states could not be called a quantum *analogue* at all. We confirm the principle on a concrete commuting pair: the number classical OT returns is the number every candidate is then obliged to match.

In [ ]:
# Two density matrices, both diagonal in the Z basis -> they commute (a classical pair).
p_vec = np.array([0.6, 0.4])
q_vec = np.array([0.1, 0.9])
rho_p = np.diag(p_vec).astype(complex)
rho_q = np.diag(q_vec).astype(complex)

print("rho_p, rho_q are both diagonal in the Z basis:")
print(f"  diag(rho_p) = {diagonal_in_computational_basis(rho_p).tolist()}")
print(f"  diag(rho_q) = {diagonal_in_computational_basis(rho_q).tolist()}")
print(f"  commutator norm ||[rho_p, rho_q]||_F = {commutativity_norm(rho_p, rho_q):.2e}"
      "   (zero -> they commute)")
print()

# On this commuting pair, classical OT is the CORRECT answer -- the consistency target.
# Ground cost on the two outcomes {0, 1}: C[i, j] = |i - j|.
cost = cityblock_cost([0, 1], [0, 1])
classical_w1 = ot.emd2(p_vec, q_vec, cost)
print(f"classical W_1 on the diagonals = {classical_w1:.4f}")
print()
print("The consistency principle: any legitimate quantum OT MUST return this same number")
print("for this commuting pair. The next notebook's coupling SDP satisfies it by")
print("construction -- the optimal quantum coupling becomes diagonal here.")

**Read the output.** The two states are diagonal in the same basis, so their commutator norm is zero to machine precision — by the gauge of `04/02`, this is a *classical* pair with no hidden coherence. On exactly this kind of pair classical OT is trustworthy: feeding the diagonals $(0.6, 0.4)$ and $(0.1, 0.9)$ to the Kantorovich solver returns $0.5000$, the cost of moving the mass that differs between them. The consistency principle now reads as a constraint with a *number* attached: whatever quantum OT we build in the rest of this module, it must return this same $0.5000$ for $\rho_p$ and $\rho_q$. That single requirement is what separates a genuine quantum analogue from an unrelated matrix functional — and it is the first test the next notebook's construction will have to pass.

## 2. Trevisan's taxonomy — the landscape of quantum OT

The consistency principle narrows the field but does not single out one winner: *many* constructions reduce to classical OT on commuting states while disagreeing with one another off it. Quantum optimal transport is therefore not a single object but a small family of non-equivalent definitions, each lifting a different facet of the classical theory. Trevisan's 2022 survey is the map of this landscape; the table below is its skeleton, with the notebook in this module where each idea appears.

| Formulation | Core idea | Where in the course |
|---|---|---|
| **Quantum couplings (SDP)** | Replace a classical coupling $P$ by a bipartite density matrix $\rho_{AB}$ whose partial traces are the two marginals; minimise $\mathrm{tr}(C\,\rho_{AB})$ (De Palma–Trevisan, 2021). | **04/04 onward** (the path this module takes) |
| **Entropic / quantum Sinkhorn** | Add a von-Neumann-entropy regulariser and iterate matrix exponentials with partial traces, exactly as classical Sinkhorn iterates a Gibbs kernel. | later in Module 4 |
| **Dynamic / Carlen–Maas** | Replace the continuity equation by a Lindblad-type generator and define a Wasserstein metric on the *dynamics* of density matrices. | later in Module 4 (brief) |
| **Channel-based** | Build a Wasserstein-like distance from the action of CPTP channels via their Stinespring dilation — no explicit coupling. | survey only |
| **Qubit-$W_1$** | Specialise the Lipschitz (Kantorovich dual) side to multi-qubit systems with a Hamming-type metric (De Palma, Marvian, Trevisan, Lloyd, 2021). | survey only |

**Different formulations, different theorems.** There is no single "*the*" quantum optimal transport, and reading these as rivals is the wrong picture — they answer different questions, and each recovers classical OT in its own commuting limit. This course commits to the **coupling SDP** as its principal object. The reason is continuity with everything you have already built: it is the most direct analogue of the Kantorovich linear program from Module 3, it inherits a clean entropic regularisation (the quantum Sinkhorn iteration), and the information-geometric reading of Module 2 — the Bures/Fisher bridge, the Amari projection picture — survives the lift. The other entries stay on the map as the wider context, exactly as Trevisan presents them.

## 3. The dictionary opens its quantum-OT row

One thread has run through the whole course: the classical ↔ quantum **dictionary**, built row by row in each module's synthesis. Module 2 filled the *information-geometry* rows — entropy, relative entropy, mutual information, the Fisher/Bures metric. Module 3 opened the *transport* rows — the coupling, the Kantorovich LP, the transportation polytope, the Birkhoff/assignment special case — and left a column deliberately blank: the quantum side of transport.

This arc is what lets us start filling that column. The consistency principle fixes the *boundary condition* (the quantum entries must reduce to the classical ones when states commute), and Trevisan's taxonomy names the *object* that goes in each cell. So the quantum-OT row of the dictionary **opens here**, with its first entries written as commitments the rest of the module will honour and verify.

| Classical | Quantum (opens here) | Where it lands |
|-----------|----------------------|----------------|
| transport plan / coupling $P \in T(a, b)$ | quantum coupling $\rho_{AB}$: bipartite, with $\mathrm{tr}_B\,\rho_{AB} = \rho_A$ and $\mathrm{tr}_A\,\rho_{AB} = \rho_B$ | 04/04 |
| Kantorovich LP $\min \langle C, P\rangle$ | quantum OT semidefinite program $\min \mathrm{tr}(C\,\rho_{AB})$ | 04/04 |
| transportation polytope $T(a, b)$ | set of quantum couplings (a convex spectrahedron) | 04/04 |
| Wasserstein-$p$ distance $W_p$ | quantum Wasserstein distance | 04/04 onward |
| **commuting / diagonal states** | **the consistency boundary: quantum OT $=$ classical OT** | **here (04/03)** |

The last row is the one this notebook contributes outright — the rest are commitments the coupling construction will discharge. With the boundary condition pinned and the object named, the dictionary's transport column is no longer blank on the quantum side; it is *open*, and the next notebook begins to fill it in.

## Your turn

**Warm-up.** Build your own commuting pair and read off its consistency target. Pick two probability vectors on three outcomes, say $a = (0.5, 0.3, 0.2)$ and $b = (0.2, 0.5, 0.3)$, form the diagonal density matrices $\rho_a = \mathrm{diag}(a)$ and $\rho_b = \mathrm{diag}(b)$, and confirm with `commutativity_norm` that they commute. Then state — *before solving* — the number any legitimate quantum OT must return for this pair, and check it with `ot.emd2` against a `cityblock_cost` on the outcomes $\{0, 1, 2\}$.

**Go further.** The consistency principle is a statement about the *commuting* boundary only. Take the non-commuting equator pair from `04/02` ($|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$) and argue, in two or three sentences, why the principle says *nothing* about the number quantum OT should return for it. Which quantity from `04/02` certifies that this pair lies strictly off the boundary?

**Challenge.** Place one formulation on the map yourself. Read the abstract of Trevisan (arXiv:2202.02091) and pick any single formulation from the taxonomy table other than the coupling SDP. In a short paragraph, state what classical object it lifts, name the one structural feature that distinguishes it from the coupling SDP, and explain how it still respects the consistency principle in its own commuting limit.

## What you built

- You turned the two negative examples of `04/01`–`04/02` into a positive law — the **diagonal-collapse consistency principle**: on commuting states, a legitimate quantum OT must reduce to classical OT, because that is exactly where the diagonal discards nothing.
- You confirmed the principle on a commuting diagonal pair, where the commutator norm is zero and classical OT returns the consistency target $0.5000$ — the number every candidate quantum OT is now obliged to match.
- You read **Trevisan's taxonomy**, the map of non-equivalent QOT formulations, and saw why this course follows the **coupling SDP**: it is the closest analogue of the Kantorovich LP and carries the entropic and information-geometric structure forward.
- You **opened the quantum-OT row** of the classical ↔ quantum dictionary, writing the consistency boundary as its first secured entry and the coupling, SDP, polytope, and distance as commitments for the rest of the module.

This is the keystone of the module's opening arc. Two notebooks ago the gap between classical and quantum transport was a puzzle on a single qubit; now it is a precise requirement with a number attached, set inside a map of the whole field. You know what a quantum OT must do, you know the family of objects that try to do it, and you know which one this course will build.

## What's next

In `04/04_quantum_coupling_sdp` the coupling formulation stops being a row in a table and becomes a concrete computation: we define the quantum coupling, set up the semidefinite program, and watch it return the consistency target on commuting states while seeing the coherence classical OT could not. The boundary condition you pinned down here is the first thing it will satisfy.

## References

- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* 22, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3
- G. De Palma, M. Marvian, D. Trevisan & S. Lloyd, "The quantum Wasserstein distance of order 1", *IEEE Transactions on Information Theory* 67, 6627–6643 (2021). DOI:10.1109/TIT.2021.3076442
- E. A. Carlen & J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability under which the Fermionic Fokker–Planck equation is gradient flow for the entropy", *Communications in Mathematical Physics* 331, 887–926 (2014). DOI:10.1007/s00220-014-2124-8

**Previous:** `notebooks/04_QuantumOptimalTransport/02_noncommuting_same_diagonal.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/04_quantum_coupling_sdp.ipynb`